<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/KG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q neo4j pandas

In [ ]:
documents = [
    {
        "title": "API Gateway",
        "text": """
        An API Gateway receives requests from clients before they reach
        the LLM serving infrastructure.

        The API Gateway can perform authentication, rate limiting,
        request validation, and request routing.

        Requests are forwarded from the API Gateway to model runners.
        """
    },

    {
        "title": "Model Runners",
        "text": """
        Model runners execute LLM inference on GPU workers.

        vLLM is an example of a model serving engine.
        Model runners can use continuous batching to improve throughput.

        vLLM uses PagedAttention to manage KV cache memory efficiently.
        """
    },

    {
        "title": "Tracing",
        "text": """
        Tracing follows an inference request across the serving system.

        A trace can follow a request through the API Gateway,
        router, model runner, and response layer.

        Metrics such as time to first token, total latency,
        throughput, and GPU utilization can be observed.
        """
    },

    {
        "title": "Fault Tolerance",
        "text": """
        Fault tolerance allows an LLM serving system to handle failures.

        Retries can handle temporary failures.
        Timeouts prevent requests from waiting indefinitely.
        Circuit breakers prevent repeated requests to unhealthy services.

        Health checks can identify unhealthy model runners.
        """
    },

    {
        "title": "Inference Routing",
        "text": """
        Inference routing decides which model runner receives a request.

        Round robin distributes requests sequentially across workers.

        Least loaded routing sends requests to workers with lower load.

        KV cache aware routing can send requests to workers that already
        contain reusable KV cache state.
        """
    }
]

In [ ]:
for doc in documents:
  print("=" * 50)
  print(doc["title"])
  print(doc["text"])

API Gateway

        An API Gateway receives requests from clients before they reach
        the LLM serving infrastructure.

        The API Gateway can perform authentication, rate limiting,
        request validation, and request routing.

        Requests are forwarded from the API Gateway to model runners.
        
Model Runners

        Model runners execute LLM inference on GPU workers.

        vLLM is an example of a model serving engine.
        Model runners can use continuous batching to improve throughput.

        vLLM uses PagedAttention to manage KV cache memory efficiently.
        
Tracing

        Tracing follows an inference request across the serving system.

        A trace can follow a request through the API Gateway,
        router, model runner, and response layer.

        Metrics such as time to first token, total latency,
        throughput, and GPU utilization can be observed.
        
Fault Tolerance

        Fault tolerance allows an LLM serving system to

In [ ]:
triples = [
    ("API Gateway", "ROUTES_TO", "Model Runner"),
    ("API Gateway", "PERFORMS", "Authentication"),
    ("API Gateway", "PERFORMS", "Rate Limiting"),
    ("API Gateway", "PERFORMS", "Request Validation"),

    ("Inference Routing", "ROUTES_TO", "Model Runner"),
    ("Round Robin", "IS_A", "Routing Strategy"),
    ("Least Loaded Routing", "IS_A", "Routing Strategy"),
    ("KV Cache Aware Routing", "IS_A", "Routing Strategy"),

    ("Model Runner", "EXECUTES", "LLM Inference"),
    ("Model Runner", "RUNS_ON", "GPU"),
    ("Model Runner", "USES", "Continuous Batching"),

    ("vLLM", "IS_A", "Model Runner"),
    ("vLLM", "USES", "PagedAttention"),

    ("PagedAttention", "MANAGES", "KV Cache"),
    ("KV Cache", "STORED_IN", "GPU Memory"),

    ("Tracing", "MEASURES", "TTFT"),
    ("Tracing", "MEASURES", "Latency"),
    ("Tracing", "MEASURES", "Throughput"),

    ("Fault Tolerance", "USES", "Retries"),
    ("Fault Tolerance", "USES", "Timeouts"),
    ("Fault Tolerance", "USES", "Circuit Breaker"),

    ("Health Check", "MONITORS", "Model Runner")
]

In [ ]:
from google.colab import userdata
from neo4j import GraphDatabase

NEO4J_URI = userdata.get("NEO4J_URI")
NEO4J_USERNAME = userdata.get("NEO4J_USERNAME")
NEO4J_PASSWORD = userdata.get("NEO4J_PASSWORD")

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j!")

Connected to Neo4j!


In [ ]:
driver.execute_query("""
    MATCH (n)
    DETACH DELETE n
""")

print("Database created")

Database created


In [ ]:
def create_relationship(subject, relation, obj):

  query = f"""
  MERGE (a:Entity {{name: $subject}})
  MERGE (b:Entity {{name: $object}})
  MERGE (a)-[:{relation}]->(b)
  """

  driver.execute_query(
      query,
      subject=subject,
      object=obj
  )

In [ ]:
for subject, relation, obj in triples:
  create_relationship(subject, relation, obj)

print("Graph created!")

Graph created!


In [ ]:
records, summary, keys = driver.execute_query("""
MATCH (a)-[r]->(b)
RETURN a.name AS source,
       type(r) AS relationship,
       b.name AS target
""")

for record in records:
    print(
        record["source"],
        "--",
        record["relationship"],
        "-->",
        record["target"]
    )

API Gateway -- ROUTES_TO --> Model Runner
API Gateway -- PERFORMS --> Authentication
API Gateway -- PERFORMS --> Rate Limiting
API Gateway -- PERFORMS --> Request Validation
Model Runner -- EXECUTES --> LLM Inference
Model Runner -- RUNS_ON --> GPU
Model Runner -- USES --> Continuous Batching
Inference Routing -- ROUTES_TO --> Model Runner
Round Robin -- IS_A --> Routing Strategy
Least Loaded Routing -- IS_A --> Routing Strategy
KV Cache Aware Routing -- IS_A --> Routing Strategy
vLLM -- IS_A --> Model Runner
vLLM -- USES --> PagedAttention
PagedAttention -- MANAGES --> KV Cache
KV Cache -- STORED_IN --> GPU Memory
Tracing -- MEASURES --> TTFT
Tracing -- MEASURES --> Latency
Tracing -- MEASURES --> Throughput
Fault Tolerance -- USES --> Retries
Fault Tolerance -- USES --> Timeouts
Fault Tolerance -- USES --> Circuit Breaker
Health Check -- MONITORS --> Model Runner


In [ ]:
records, _, _ = driver.execute_query("""
MATCH (a:Entity {name: $name})-[r]-(b)
RETURN
    a.name AS source,
    type(r) AS relation,
    b.name AS target
""",
name="vLLM"
)

for r in records:
    print(
        r["source"],
        r["relation"],
        r["target"]
    )

vLLM IS_A Model Runner
vLLM USES PagedAttention


In [ ]:
def get_graph_context(entity, max_hops=3):

    query = """
    MATCH p=(start:Entity {name: $entity})-[*1..3]-(connected)
    UNWIND relationships(p) AS rel

    WITH
        startNode(rel) AS source,
        rel,
        endNode(rel) AS target

    RETURN DISTINCT
        source.name AS source,
        type(rel) AS relationship,
        target.name AS target
    LIMIT 50
    """

    records, _, _ = driver.execute_query(
        query,
        entity=entity
    )

    context = []

    for r in records:
        context.append(
            f"{r['source']} --{r['relationship']}--> {r['target']}"
        )

    return context

In [ ]:
context = get_graph_context("vLLM")

for line in context:
    print(line)

vLLM --IS_A--> Model Runner
API Gateway --ROUTES_TO--> Model Runner
API Gateway --PERFORMS--> Authentication
API Gateway --PERFORMS--> Rate Limiting
API Gateway --PERFORMS--> Request Validation
Inference Routing --ROUTES_TO--> Model Runner
Model Runner --EXECUTES--> LLM Inference
Model Runner --RUNS_ON--> GPU
Model Runner --USES--> Continuous Batching
Health Check --MONITORS--> Model Runner
vLLM --USES--> PagedAttention
PagedAttention --MANAGES--> KV Cache
KV Cache --STORED_IN--> GPU Memory


In [ ]:
known_entities = [
    subject for subject, _, _ in triples
] + [
    obj for _, _, obj in triples
]

known_entities = list(set(known_entities))

known_entities

['Tracing',
 'GPU Memory',
 'Fault Tolerance',
 'LLM Inference',
 'Round Robin',
 'Authentication',
 'Retries',
 'PagedAttention',
 'Least Loaded Routing',
 'Latency',
 'API Gateway',
 'Routing Strategy',
 'vLLM',
 'GPU',
 'Circuit Breaker',
 'Request Validation',
 'Inference Routing',
 'Continuous Batching',
 'TTFT',
 'Model Runner',
 'Health Check',
 'Rate Limiting',
 'KV Cache Aware Routing',
 'Timeouts',
 'Throughput',
 'KV Cache']

In [ ]:
def detect_entities(question):

    matches = []

    for entity in known_entities:
        if entity.lower() in question.lower():
            matches.append(entity)

    return matches

In [ ]:
question = "How does vLLM manage GPU memory?"

entities = detect_entities(question)

print(entities)

['GPU Memory', 'vLLM', 'GPU']
